# 03 · Advanced AI — Cortex Search  *(stretch)*

**Optional.** If you're short on time, skip straight to `04`. Nothing later
depends on this.

Stage 2's `AI_CLASSIFY` is great for *known* buckets (dashboards). Cortex Search
is for the *unknown* — "find feedback like this one" and emerging themes — with a
managed, auto-embedded vector index. No hand-built `AI_EMBED` needed.

### Context

In [ ]:
SET sch = 'PLG_CORTEX_WORKSHOP.WS_' || REGEXP_REPLACE(CURRENT_USER(), '[^A-Za-z0-9_]', '_');
USE WAREHOUSE PLG_WORKSHOP_WH;
USE SCHEMA IDENTIFIER($sch);

### 1. Create the search service over the free text
Attributes let you filter results by brand, topic, sentiment, etc.

In [ ]:
CREATE OR REPLACE CORTEX SEARCH SERVICE SURVEY_FEEDBACK_SEARCH
  ON clarity_comment
  ATTRIBUTES brand, email_type, clarity_sentiment, clarity_topic
  WAREHOUSE = PLG_WORKSHOP_WH
  TARGET_LAG = '1 hour'
AS
  SELECT response_id, clarity_comment, brand, email_type, clarity_sentiment, clarity_topic
  FROM SURVEY_ENRICHED
  WHERE clarity_comment IS NOT NULL;

### 2. Find feedback like this — filtered to negative sentiment

In [ ]:
SELECT PARSE_JSON(
  SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
    'SURVEY_FEEDBACK_SEARCH',
    '{
       "query": "de e-mail was verwarrend en te lang",
       "columns": ["clarity_comment","brand","clarity_topic"],
       "filter": {"@eq": {"clarity_sentiment": "negative"}},
       "limit": 5
     }'
  )
) AS results;

### Checkpoint ✅
You get relevant similar comments, filtered by an attribute. Note the contrast:
- **`AI_CLASSIFY` (Stage 2)** → fixed taxonomy, great for counting/dashboards.
- **Cortex Search (here)** → open-ended discovery, great for "what are people
  suddenly complaining about?"

Search is *retrieval*, not aggregation — you'd still use the semantic view for
"how many / what's the average". Next: `04_consumption_eval.ipynb`.